# Remote GPU setup (SETUP-05)

Run this notebook on Colab for the GPU-heavy tasks (task 3 UVR, task 4 Food-101).
Before running: **Runtime > Change runtime type > GPU**. With the Google One
student plan, Colab gives more compute units than the free tier (better GPU
priority, e.g. T4/L4/A100 depending on availability, and longer sessions)
but sessions can still disconnect/time out, so checkpoints must live on a
persistent storage (Google Drive here), not Colab's local disk.

This notebook:
1. reports the GPU actually assigned to the session,
2. mounts Drive and clones the repo + data,
3. proves a training run can be interrupted and resumed without losing
   progress, using `common/checkpointing.py`.

## 1. GPU model available

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Persistent storage (Google Drive) for checkpoints

Colab's local disk is wiped when the session ends, so checkpoints go to
Drive instead.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import os

CHECKPOINT_DIR = "/content/drive/MyDrive/ml_tek5_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 3. Clone the repo and pull in the extra (GPU-task) dependencies

Colab already ships a CUDA-matched PyTorch build; reinstalling torch here
(e.g. via `uv sync`, which pins the CPU wheel for the no-GPU laptop) would
throw that away. Only the packages Colab doesn't already have are installed,
straight from `pyproject.toml`.

The repo is **private**, so cloning over HTTPS needs a GitHub token (a plain
`git clone` fails with `could not read Username ... No such device or
address`, since Colab can't prompt interactively for credentials). Create a
[fine-grained personal access token](https://github.com/settings/tokens)
with read-only `Contents` access to this repo, then either store it as a
Colab secret named `GITHUB_TOKEN` (recommended: toggle notebook access on)
or paste it when prompted below.

In [ ]:
import base64
import getpass

try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if not token:
    token = getpass.getpass(
        "GitHub token (repo scope) for the private "
        "NattanCochet/G-AIA-900-PAR-9-1-machinelearning-11 repo: "
    )

REPO_URL = "https://github.com/NattanCochet/G-AIA-900-PAR-9-1-machinelearning-11.git"
REPO_DIR = "/content/ml-tek5-project"
# GitHub's git-over-HTTPS endpoint only accepts Basic auth (unlike the REST
# API, which also accepts Bearer) -- same scheme as actions/checkout. Passed
# as a one-off git config flag instead of embedding the token in the URL, so
# it never ends up persisted in {REPO_DIR}/.git/config.
basic_auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
AUTH_HEADER = f"AUTHORIZATION: basic {basic_auth}"

# Clear any extraheader Colab (or a previous run) may already have
# configured for github.com -- combined with ours below it would send two
# "Authorization" headers, which GitHub rejects with 400 "Duplicate
# header: 'Authorization'". Also disable the credential helper so it can't
# inject a second one either.
!git config --global --unset-all http.https://github.com/.extraheader 2>/dev/null
!git config --system --unset-all http.https://github.com/.extraheader 2>/dev/null

if not os.path.isdir(REPO_DIR):
    !git -c credential.helper= -c http.extraheader="{AUTH_HEADER}" clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} config --local --unset-all http.https://github.com/.extraheader 2>/dev/null
    !git -C {REPO_DIR} -c credential.helper= -c http.extraheader="{AUTH_HEADER}" pull

In [ ]:
# Task 3 (UVR) + task 4 (Food-101) extras not already on Colab.
# Keep this list in sync with pyproject.toml's dependencies.
!pip install -q soundfile onnx onnxruntime onnx2pytorch mir_eval \
    transformers datasets accelerate timm

In [ ]:
import importlib
import sys

sys.path.insert(0, REPO_DIR)  # so `import common` resolves, as from the repo root
importlib.invalidate_caches()  # in case REPO_DIR was cached as empty by an earlier failed clone

import torch

assert torch.cuda.is_available(), "GPU not enabled: Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

## 4. Checkpoint save/resume demo

Two ways to run this:
- **Quick check** (same session): run every cell below top to bottom.
  It proves the save/load logic is correct, but doesn't prove a real
  Colab disconnect is survived.
- **Real manual test** (recommended at least once): run "Shared demo
  setup" then "Part A", then **actually restart the runtime**
  (Runtime > Restart session, or for a harsher test, Runtime >
  Disconnect and delete runtime, which wipes the VM entirely), then
  re-run sections 1-3 above (GPU check, Drive mount, clone+install)
  plus "Shared demo setup", and finally run only "Part B" (not Part A
  again). Only the checkpoint and the reference-losses file on Drive
  survive a full disconnect; Part B loads both from there, so if it
  still matches you've proven the setup survives a real session loss,
  not just a simulated one.

`ModuleNotFoundError: No module named 'common'` below means section 3
(clone + `sys.path`) wasn't (re-)run in this runtime; "Shared demo
setup" re-adds the repo to `sys.path` itself as a fallback, but the repo
still has to actually be cloned to `/content/ml-tek5-project` first.

### Shared demo setup (re-run this after restarting too)

In [ ]:
import importlib
import sys

# Fallback in case this cell runs before section 3's sys.path.insert,
# e.g. right after a runtime restart.
REPO_DIR = globals().get("REPO_DIR", "/content/ml-tek5-project")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()  # in case REPO_DIR was cached as empty by an earlier failed clone

import torch
from torch import nn

from common import set_seed
from common.checkpointing import load_checkpoint, save_checkpoint

device = torch.device("cuda")
ckpt_path = f"{CHECKPOINT_DIR}/demo_checkpoint.pt"
reference_path = f"{CHECKPOINT_DIR}/demo_reference_losses.json"


def make_model_and_opt():
    set_seed(0)
    model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 1)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    return model, opt


set_seed(0)
x = torch.randn(64, 10, device=device)
y = torch.randn(64, 1, device=device)
loss_fn = nn.MSELoss()


def run(model, opt, start_epoch, end_epoch, save=False):
    losses = []
    for epoch in range(start_epoch, end_epoch):
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if save:
            save_checkpoint(ckpt_path, model, opt, epoch)
    return losses

### Part A — run now

Computes the uninterrupted reference run, then trains epochs 0-4
while checkpointing every epoch, and saves both to Drive so Part B
can find them even after a real restart.

In [ ]:
import json

model_a, opt_a = make_model_and_opt()
straight_losses = run(model_a, opt_a, 0, 10)

model_b, opt_b = make_model_and_opt()
first_half = run(model_b, opt_b, 0, 5, save=True)

with open(reference_path, "w") as f:
    json.dump({"straight": straight_losses, "first_half": first_half}, f)

print("straight  :", [round(v, 6) for v in straight_losses])
print("first_half:", [round(v, 6) for v in first_half])
print("\nNow restart the runtime (Runtime menu), then re-run cells")
print("1-3 and 'Shared demo setup' above, and run Part B below.")

### Part B — run after restarting the runtime

Do **not** re-run Part A first (that would start training over from
epoch 0 and defeat the test). This loads the checkpoint and the
reference losses purely from Drive, as if this were a brand new
session, and continues training from epoch 5.

In [ ]:
import json

with open(reference_path) as f:
    reference = json.load(f)

model_c, opt_c = make_model_and_opt()
next_epoch, _ = load_checkpoint(ckpt_path, model_c, opt_c, map_location=device)
assert next_epoch == 5, next_epoch
second_half = run(model_c, opt_c, next_epoch, 10)

resumed_losses = reference["first_half"] + second_half
print("straight:", [round(v, 6) for v in reference["straight"]])
print("resumed :", [round(v, 6) for v in resumed_losses])
assert resumed_losses == reference["straight"], "resume lost progress / diverged!"
print("OK: training resumed after the restart with zero lost progress")

## 5. Using this in a real task 3 / task 4 training script

```python
from common.checkpointing import save_checkpoint, load_checkpoint

start_epoch = 0
if os.path.exists(ckpt_path):
    start_epoch, extra = load_checkpoint(ckpt_path, model, optimizer, map_location=device)

for epoch in range(start_epoch, total_epochs):
    train_one_epoch(...)
    save_checkpoint(ckpt_path, model, optimizer, epoch)
```

Keep `ckpt_path` under `CHECKPOINT_DIR` (Drive), and re-run this notebook's
clone+install cells from the top the next time the session disconnects; the
training loop will pick up where it left off.